# 01 — Data Acquisition and Validation

**Project**: LLM-Based Startup Risk Profiling and Funding Outcome Prediction  
**Author**: ADS  
**Date**: October 2026

This notebook covers:
1. Dataset loading from CSV
2. Schema inspection and data types
3. Missing values, duplicates, invalid values
4. Target encoding (binary: acquired/IPO = success, closed = failure)
5. Data quality report

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw_data, encode_target, parse_dates, clean_funding_total
from src.validation import validate_all

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

import warnings
warnings.filterwarnings('default')

## 1. Load Raw Data

In [ ]:
DATA_PATH = '../data/startup_data.csv'
df = load_raw_data(DATA_PATH)
df.head()

## 2. Inspect Schema

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nDtypes:\n{df.dtypes}')

## 3. Clean Data

In [ ]:
df = clean_funding_total(df)
df = parse_dates(df)
df = encode_target(df)
df.head()

## 4. Validation Report

In [ ]:
report = validate_all(df)

## 5. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df['status'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Original Status Distribution')
axes[0].tick_params(axis='x', rotation=45)

df_model = df.dropna(subset=['funding_success'])
df_model['funding_success'].value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Binary Target Distribution (1=Success, 0=Failure)')
axes[1].set_xticklabels(['Failure', 'Success'], rotation=0)

plt.tight_layout()
plt.savefig('../reports/figures/target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Missing Values Heatmap

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Values Pattern')
plt.tight_layout()
plt.savefig('../reports/figures/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Cleaned Data

In [ ]:
df_model.to_csv('../data/startup_cleaned.csv', index=False)
print(f'Saved {len(df_model)} rows to data/startup_cleaned.csv')